# Data Acquisition and Vectorization

This notebook demonstrates how to:
1. Connect to Confluence and retrieve project documentation
2. Extract GitHub links and retrieve repository information
3. Process and chunk the data
4. Generate embeddings and store in vector database

In [ ]:
import sys
sys.path.append('..')

from src.config import ConfigConfluenceRag
from src.confluence.client import ConfluenceClient
from src.confluence.parser import ConfluenceParser
from src.github.client import GitHubClient
from src.github.parser import GitHubParser
from src.rag.vectorstore import VectorStore
from src.rag.embeddings import EmbeddingManager
from loguru import logger
import pandas as pd


/Users/bruengw/Library/CloudStorage/OneDrive-AbbVieInc(O365)/Documents/confluence_rag/.conf_rag/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Validate Configuration

In [12]:
ConfigConfluenceRag = ConfigConfluenceRag
if not ConfigConfluenceRag.validate():
    raise ValueError("Configuration validation failed. Please check your .env file.")

print("✅ Configuration validated successfully")
print(f"Confluence URL: {ConfigConfluenceRag.CONFLUENCE_URL}")
print(f"Confluence Username: {ConfigConfluenceRag.CONFLUENCE_USERNAME}")
print(f"Confluence API Token: {ConfigConfluenceRag.CONFLUENCE_API_TOKEN}")
print(f"Confluence Space: {ConfigConfluenceRag.CONFLUENCE_SPACE_KEY}")
print(f"Embedding Model: {ConfigConfluenceRag.EMBEDDING_MODEL}")
print(f"Vector DB Path: {ConfigConfluenceRag.VECTOR_DB_PATH}")

2026-01-22 09:54:00.105 | INFO     | src.config:validate:59 - All required configurations are present


✅ Configuration validated successfully
Confluence URL: https://confluence.abbvienet.com/
Confluence Username: garrick.bruening@abbvie.com
Confluence API Token: OTcyMDM2OTcxOTMyOqBkBzNmGpsNcmW5TaIjhEg2qt4w
Confluence Space: DSA
Embedding Model: all-MiniLM-L6-v2
Vector DB Path: ./Data_Storage/vector_db


## 2. Retrieve Confluence Pages

In [ ]:
import config
from atlassian.confluence import ConfluenceCloud, ConfluenceServer

test = ConfluenceServer(
    url='https://confluence.abbvienet.com',
    username='garrick.bruening@abbvie.com',
    password=ConfigConfluenceRag.CONFLUENCE_API_TOKEN,
    cloud=False,
    verify_ssl=False
)

space_DSA = test.get_space('DSA', expand='description.plain,homepage')
print(space_DSA)


HTTPError: Unauthorized (401)

In [11]:
import requests

API_URL = 'https://confluence.abbvienet.com'
API_TOKEN = f'{ConfigConfluenceRag.CONFLUENCE_API_TOKEN}'
EMAIL = "garrick.bruening@abbvie.com"

# headers = {
#     "Accept": "application/json",
#     "Content-Type": "application/json"
# }

auth = (EMAIL, API_TOKEN)

try:
    response = requests.get(f"{API_URL}/space", auth=auth)
    response.raise_for_status() # Raises an exception for bad status codes
    spaces = response.json()
    print("Spaces:", spaces)
except requests.exceptions.RequestException as e:
    print(f"Error connecting to Confluence: {e}")

Error connecting to Confluence: HTTPSConnectionPool(host='confluence.abbvienet.com', port=443): Max retries exceeded with url: /space (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1081)')))


In [28]:
space_info = test.get_space('Data Science and Analytics Home', expand='description.plain,homepage')
print(space_info)

HTTPError: Unauthorized (401)

In [25]:
spaces = test.get_all_spaces(limit=1)

HTTPError: Unauthorized (401)

In [ ]:
test.get_all_spaces(start=1, limit=20, expand=None)

SSLError: HTTPSConnectionPool(host='confluence.abbvienet.com', port=443): Max retries exceeded with url: /rest/api/space?limit=20 (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1081)')))

In [6]:
# Initialize Confluence client
confluence_client = ConfluenceClient(
    url=config.CONFLUENCE_URL,
    username=config.CONFLUENCE_USERNAME,
    api_token=config.CONFLUENCE_API_TOKEN,
    space_key=config.CONFLUENCE_SPACE_KEY
)

print("Retrieving Confluence pages...")
pages_content = confluence_client.get_all_pages_content()
print(f"Retrieved {len(pages_content)} pages from Confluence")

2026-01-13 12:11:37.700 | INFO     | src.confluence.client:__init__:33 - Initialized Confluence client for space: DSA
2026-01-13 12:11:37.701 | INFO     | src.confluence.client:get_all_pages_content:115 - Retrieving full content for all pages
2026-01-13 12:11:37.702 | INFO     | src.confluence.client:get_all_pages:45 - Retrieving pages from space: DSA
2026-01-13 12:11:37.899 | ERROR    | src.confluence.client:get_all_pages:70 - Error retrieving pages from Confluence: HTTPSConnectionPool(host='confluence.abbvienet.com', port=443): Max retries exceeded with url: /rest/api/content?spaceKey=DSA&limit=500&expand=version&type=page (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1081)')))


Retrieving Confluence pages...


SSLError: HTTPSConnectionPool(host='confluence.abbvienet.com', port=443): Max retries exceeded with url: /rest/api/content?spaceKey=DSA&limit=500&expand=version&type=page (Caused by SSLError(SSLCertVerificationError(1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: self-signed certificate in certificate chain (_ssl.c:1081)')))

In [ ]:
# Display sample page information
if pages_content:
    sample_page = pages_content[0]
    print(f"Sample Page Title: {sample_page['title']}")
    print(f"URL: {sample_page['url']}")
    print(f"Content Length: {len(sample_page['content'])} characters")

## 3. Parse Confluence Content

In [ ]:
# Initialize parser
confluence_parser = ConfluenceParser()

# Parse all pages
print("Parsing Confluence pages...")
parsed_pages = confluence_parser.parse_pages(pages_content)
print(f"Parsed {len(parsed_pages)} pages successfully")

In [ ]:
# Display page statistics
df_pages = pd.DataFrame([
    {
        'title': page['title'],
        'text_length': len(page['text']),
        'num_links': len(page['links']),
        'num_headers': len(page['headers']),
        'num_tables': len(page['tables'])
    }
    for page in parsed_pages
])

print("\nPage Statistics:")
print(df_pages.describe())
df_pages.head()

## 4. Extract GitHub Links and Retrieve Repositories

In [ ]:
# Extract GitHub links from pages
print("Extracting GitHub links...")
github_links = []

for page in pages_content:
    links = confluence_client.extract_github_links(page['content'])
    for link in links:
        github_links.append({
            'page_title': page['title'],
            'page_url': page['url'],
            'github_url': link
        })

print(f"Found {len(github_links)} GitHub links across {len(set(link['github_url'] for link in github_links))} unique repositories")

In [ ]:
# Initialize GitHub client
github_client = GitHubClient(token=config.GITHUB_TOKEN)
github_parser = GitHubParser()

# Retrieve repository information
github_summaries = []
unique_repos = list(set(link['github_url'] for link in github_links))

print(f"\nRetrieving information for {len(unique_repos)} repositories...")
for repo_url in unique_repos:
    try:
        summary = github_client.get_repository_summary(repo_url)
        if summary:
            parsed_summary = github_parser.parse_repository_summary(summary)
            github_summaries.append(parsed_summary)
            print(f"✓ {summary['info']['full_name']}")
    except Exception as e:
        print(f"✗ Failed to retrieve {repo_url}: {str(e)}")

print(f"\nSuccessfully retrieved {len(github_summaries)} repository summaries")

## 5. Chunk Documents for Vectorization

In [ ]:
# Chunk Confluence pages
confluence_chunks = []

for page in parsed_pages:
    text_chunks = confluence_parser.chunk_text(
        page['text'],
        chunk_size=config.CHUNK_SIZE,
        chunk_overlap=config.CHUNK_OVERLAP
    )
    
    for i, chunk in enumerate(text_chunks):
        confluence_chunks.append({
            'text': chunk,
            'metadata': {
                'title': page['title'],
                'url': page['url'],
                'source_type': 'confluence',
                'chunk_id': i,
                'space': page['space']
            }
        })

print(f"Created {len(confluence_chunks)} chunks from Confluence pages")

In [ ]:
# Create chunks from GitHub summaries
github_chunks = []

for summary in github_summaries:
    text_summary = github_parser.create_text_summary(summary)
    
    github_chunks.append({
        'text': text_summary,
        'metadata': {
            'title': summary['repository_name'],
            'url': summary['url'],
            'source_type': 'github',
            'language': summary['language']
        }
    })

print(f"Created {len(github_chunks)} chunks from GitHub repositories")

In [ ]:
# Combine all chunks
all_chunks = confluence_chunks + github_chunks
print(f"\nTotal chunks to vectorize: {len(all_chunks)}")

## 6. Generate Embeddings and Store in Vector Database

In [ ]:
# Initialize embedding manager
embedding_manager = EmbeddingManager(model_name=config.EMBEDDING_MODEL)
print(f"Embedding model: {embedding_manager.model_name}")
print(f"Embedding dimension: {embedding_manager.embedding_dimension}")

In [ ]:
# Initialize vector store
vector_store = VectorStore(
    persist_directory=config.VECTOR_DB_PATH,
    collection_name='confluence_docs'
)

# Clear existing data (optional)
if vector_store.count() > 0:
    print(f"Clearing existing {vector_store.count()} documents from vector store...")
    vector_store.clear_collection()
    print("Vector store cleared")

In [ ]:
# Prepare data for storage
texts = [chunk['text'] for chunk in all_chunks]
metadatas = [chunk['metadata'] for chunk in all_chunks]
ids = [f"doc_{i}" for i in range(len(all_chunks))]

# Add documents to vector store
print("\nAdding documents to vector store...")
vector_store.add_documents(texts=texts, metadatas=metadatas, ids=ids)

print(f"✅ Successfully added {vector_store.count()} documents to vector store")

## 7. Verify Vector Store

In [ ]:
# Test query
test_query = "What data science projects are documented?"
print(f"Test query: {test_query}\n")

results = vector_store.query(query_text=test_query, n_results=3)

print("Top 3 results:")
for i, (doc, meta, dist) in enumerate(zip(results['documents'], results['metadatas'], results['distances']), 1):
    print(f"\n{i}. Title: {meta['title']}")
    print(f"   Source: {meta['source_type']}")
    print(f"   Distance: {dist:.4f}")
    print(f"   Preview: {doc[:200]}...")

## Summary

This notebook has:
- ✅ Retrieved and parsed Confluence pages
- ✅ Extracted GitHub links and retrieved repository information
- ✅ Chunked documents for optimal retrieval
- ✅ Generated embeddings using sentence transformers
- ✅ Stored documents in ChromaDB vector database

The vector database is now ready for use with the RAG pipeline!